# LAMP-Forge end-to-end walkthrough

**Goal of this notebook:** run LAMP-Forge stage-by-stage so you can see what each step produces, inspect intermediate results, and tweak parameters interactively.

**Target:** *Mycobacterium tuberculosis* `rpoB`. Same target as `config/example_config.yaml`; we'll re-do the whole pipeline in Python rather than via the CLI so each step's output is visible inline.

**Prereqs:**

- LAMP-Forge installed (`pip install -e ".[dev]"` from repo root)
- `mafft` and `ncbi-blast+` on PATH (or run this from the `lamp-forge-jupyter` Docker profile)
- Optional: `NCBI_EMAIL` and `NCBI_API_KEY` env vars set
- 2-3 off-target FASTA files in `../input/off_targets/` (see `docs/cookbook.md` for what to put there)

**Runtime:** ~3-5 minutes the first time (mostly Entrez fetch + MAFFT). Subsequent runs hit the cache and complete in <30 seconds.

## 0. Setup


In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lamp_forge import align, conserve, fetch, primer_design, report, specificity
from lamp_forge.config import load_config

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

WORK = Path('walkthrough_results')
WORK.mkdir(exist_ok=True)
print(f'Working in: {WORK.resolve()}')

## 1. Load config

We'll use the bundled example config and patch the output directory + email so the notebook can run anywhere.

In [ ]:
import os

config = load_config('../config/example_config.yaml')
config.output_dir = WORK
if os.environ.get('NCBI_EMAIL'):
    config.email = os.environ['NCBI_EMAIL']
print(f'Target: {config.target_name}  (taxon {config.taxon_id}, gene {config.gene})')
print(f'Conservation: entropy ≤ {config.entropy_threshold} bits over ≥{config.min_region_length}bp windows')
print(f'Primer Tm: {config.tm_min}-{config.tm_max}°C, spread ≤ {config.tm_match_tolerance}°C')

## 2. Fetch sequences from NCBI

First run: hits Entrez, caches results. Re-runs: served from the cache.

In [ ]:
records = fetch.fetch_for_config(config)
print(f'Retrieved {len(records)} sequences')
for r in records[:5]:
    print(f'  {r.id}  ({len(r.seq)} bp)  {r.description[:80]}')
raw_fasta = WORK / 'sequences.fasta'
fetch.write_fasta(records, raw_fasta)

## 3. Multiple sequence alignment (MAFFT)

In [ ]:
aligned_fasta = WORK / 'alignment.fasta'
msa = align.align_records(raw_fasta, aligned_fasta)
print(f'Alignment: {len(msa)} sequences × {msa.get_alignment_length()} bp')

## 4. Conservation track

Per-position Shannon entropy + window smoothing.

In [ ]:
track = conserve.compute_track(msa, window_size=config.window_size)

fig, ax = plt.subplots(figsize=(14, 3.5))
x = np.arange(len(track.smoothed_entropy))
ax.plot(x, track.raw_entropy, color='#94a3b8', linewidth=0.5, label='raw entropy')
ax.plot(x, track.smoothed_entropy, color='#0369a1', linewidth=1.2, label='smoothed (window=30)')
ax.axhline(config.entropy_threshold, color='red', linestyle='--', linewidth=0.8, label=f'threshold = {config.entropy_threshold}')
ax.set_xlabel('Alignment position (bp)')
ax.set_ylabel('Shannon entropy (bits)')
ax.set_title('Per-position conservation across the M. tuberculosis rpoB alignment')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
regions = conserve.find_conserved_regions(
    track,
    entropy_threshold=config.entropy_threshold,
    min_region_length=config.min_region_length,
)
print(f'Found {len(regions)} conserved region(s) ≥{config.min_region_length}bp:')
for r in regions:
    print(f'  {r.region_id}: positions {r.start}-{r.end} ({r.length} bp), mean entropy {r.mean_entropy:.3f} bits')

## 5. Primer design

For each conserved region, generate candidate F3/B3/FIP/BIP (+ LF/LB) sets satisfying LAMP geometry and primer3 thermodynamic constraints.

In [ ]:
primer_sets = primer_design.design_all(regions, config, max_sets_per_region=30)
print(f'Designed {len(primer_sets)} primer sets across {len(regions)} region(s)')

# Quick look at the top 5 sets pre-specificity (composite is conservation × thermo here)
rows = []
for s in primer_sets[:5]:
    rows.append({
        'set_id': s.set_id,
        'region': s.region_id,
        'mean_tm': s.mean_tm,
        'tm_spread': s.tm_spread,
        'f2_b2': s.f2_b2_distance,
        'has_loops': s.has_loop_primers,
        'conservation': s.conservation_score,
        'thermo': s.thermodynamic_score,
    })
pd.DataFrame(rows)

## 6. Specificity screen

Build a local BLAST database from any off-target FASTAs in `input/off_targets/` (or skip this step if you don't have any yet).

In [ ]:
off_target_dir = Path('../input/off_targets')
off_targets = specificity.discover_off_targets(off_target_dir) if off_target_dir.exists() else []
print(f'Off-target genomes available: {len(off_targets)}')
if off_targets:
    db = WORK / 'off_targets_db'
    specificity.build_blast_db(off_targets, db)
    specificity.screen_all(primer_sets, db, config)
    print('Specificity scoring complete.')
else:
    print('Skipping specificity screen — drop FASTAs into input/off_targets/ to enable it.')

## 7. Inspect the top primer set


In [ ]:
if primer_sets:
    top = primer_sets[0]
    print(f'Top set: {top.set_id}  (region {top.region_id}, composite {top.composite_score:.3f})')
    print(f'  Conservation: {top.conservation_score:.3f}')
    print(f'  Specificity:  {top.specificity_score:.3f}')
    print(f'  Thermodynamic: {top.thermodynamic_score:.3f}')
    print(f'  F2-B2 inner amplicon: {top.f2_b2_distance} bp')
    print(f'  Mean Tm: {top.mean_tm:.1f}°C, spread: {top.tm_spread:.1f}°C')
    print()
    for p in top.primers:
        print(f'  {p.role:>3}: {p.sequence}  (Tm {p.tm:.1f}°C, GC {p.gc_percent:.0f}%, len {p.length})')
    if top.warnings:
        print()
        for w in top.warnings:
            print(f'  WARNING: {w}')

## 8. Write the full reports


In [ ]:
report.write_json(primer_sets, WORK / 'primer_sets.json')
report.write_csv(primer_sets, WORK / 'primer_sets.csv')
report.write_conservation_tsv(track, WORK / 'conservation.tsv')
report.render_html(
    primer_sets, regions, track, config,
    n_input_seqs=len(records),
    n_off_targets=len(off_targets),
    path=WORK / 'lamp_forge_report.html',
)
print(f'Outputs written to {WORK.resolve()}/')
for f in sorted(WORK.iterdir()):
    if f.is_file():
        print(f'  {f.name}  ({f.stat().st_size:,} bytes)')

## 9. Next steps

Open `lamp_forge_report.html` in your browser to see the full ranked output with plots.

Things to try from here:

- **Tighten conservation** — drop `entropy_threshold` to 0.10 and re-run from step 4. You'll get fewer but tighter regions.
- **Add more off-targets** — drop another close relative into `input/off_targets/` and re-run from step 6. Watch the specificity scores shift.
- **Compare two genes** — change `gene: rpoB` to `gene: gyrB` in the config and see whether *M. tuberculosis* yields a cleaner conservation profile in a different marker.
- **Try a different organism** — see [`docs/cookbook.md`](../docs/cookbook.md) for ready-to-run configs for *Salmonella*, SARS-CoV-2, *Plasmodium*, and KPC carbapenemase.